1-Detección de duplicados

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../data/spotify_procesado00.csv")

print(df.duplicated().sum())

450


In [3]:
df = df.drop_duplicates()

print(df.shape)

df.to_csv("../data/spotify_procesado01.csv", index=False)

(113549, 20)


2-Verificación de datos

In [4]:
df = pd.read_csv("../data/spotify_procesado01.csv")

print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 113549 entries, 0 to 113548
Data columns (total 20 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   track_id          113549 non-null  str    
 1   artists           113549 non-null  str    
 2   album_name        113549 non-null  str    
 3   track_name        113549 non-null  str    
 4   popularity        113549 non-null  int64  
 5   duration_ms       113549 non-null  int64  
 6   explicit          113549 non-null  bool   
 7   danceability      113549 non-null  float64
 8   energy            113549 non-null  float64
 9   key               113549 non-null  int64  
 10  loudness          113549 non-null  float64
 11  mode              113549 non-null  int64  
 12  speechiness       113549 non-null  float64
 13  acousticness      113549 non-null  float64
 14  instrumentalness  113549 non-null  float64
 15  liveness          113549 non-null  float64
 16  valence           113549 non-nu

3- Preparación

eliminar los registros que no han sido reproducidos con popularidad 0

In [8]:
df = df[df["popularity"] > 0].copy()
df.info()

<class 'pandas.DataFrame'>
Index: 97706 entries, 0 to 113548
Data columns (total 23 columns):
 #   Column            Non-Null Count  Dtype   
---  ------            --------------  -----   
 0   track_id          97706 non-null  str     
 1   artists           97706 non-null  str     
 2   album_name        97706 non-null  str     
 3   track_name        97706 non-null  str     
 4   popularity        97706 non-null  int64   
 5   duration_ms       97706 non-null  int64   
 6   explicit          97706 non-null  bool    
 7   danceability      97706 non-null  float64 
 8   energy            97706 non-null  float64 
 9   key               97706 non-null  int64   
 10  loudness          97706 non-null  float64 
 11  mode              97706 non-null  int64   
 12  speechiness       97706 non-null  float64 
 13  acousticness      97706 non-null  float64 
 14  instrumentalness  97706 non-null  float64 
 15  liveness          97706 non-null  float64 
 16  valence           97706 non-null  flo

resultado: eliminados resultados que distorsionan cualquier conclusión posible. 

In [9]:
df = df[df["popularity"] > 0].copy()

def clasificar_macro_genero(genre):
    g = str(genre).lower()
    if any(term in g for term in ["hip-hop", "rap", "reggaeton", "latin", "trap", "urban"]):
        return "Urbano / Latino"
    elif any(term in g for term in ["rock", "metal", "grunge", "punk", "alternative", "psych-rock"]):
        return "Rock / Alternativo"
    elif any(term in g for term in ["house", "techno", "electro", "dance", "edm", "trance", "progressive-house"]):
        return "Electrónica / Dance"
    elif any(term in g for term in ["pop", "k-pop", "mandopop", "pop-film"]):
        return "Pop / Comercial"
    elif any(term in g for term in ["chill", "ambient", "acoustic", "piano", "sad", "emo"]):
        return "Introspectivo / Acústico"
    else:
        return "Otros / Raíz / Tradicional"

df["macro_genre"] = df["track_genre"].apply(clasificar_macro_genero)

print(df.groupby("macro_genre").agg({
    "popularity": "mean",
    "energy": "mean",
    "valence": "mean",
    "explicit": "mean",
    "track_id": "count"
}).sort_values(by="popularity", ascending=False))

                            popularity    energy   valence  explicit  track_id
macro_genre                                                                   
Introspectivo / Acústico     50.412833  0.425490  0.361542  0.200071      5673
Pop / Comercial              47.450727  0.625527  0.503571  0.031628      8189
Rock / Alternativo           41.631092  0.777851  0.463789  0.097922     13521
Urbano / Latino              40.051992  0.716881  0.607564  0.225157      2385
Electrónica / Dance          39.036531  0.728431  0.440293  0.106265     11716
Otros / Raíz / Tradicional   35.459784  0.619679  0.475820  0.069795     56222


In [10]:




df["dissonance"] = df["energy"] - df["valence"]

urbano = df["track_genre"].str.contains("hip-hop|rap|reggaeton|latin|trap", case=False, na=False)
df["es_urbano"] = urbano

print("=== 1. Impacto del Contenido Explícito en el Mercado Activo ===")
print(df.groupby("explicit")[["popularity", "energy", "valence", "dissonance", "danceability"]].mean())

print("\n=== 2. Comparativa Estricta: Urbano vs Resto (Mercado Activo) ===")
print(df.groupby("es_urbano")[["popularity", "energy", "valence", "dissonance", "explicit", "danceability"]].mean())

print("\n=== 3. Relación de la Dissonancia con la Popularidad (Cuartiles) ===")
df["dissonance_tier"] = pd.qcut(df["dissonance"], q=4, labels=["Baja Tensión", "Media-Baja", "Media-Alta", "Máxima Tensión"])
print(df.groupby("dissonance_tier")["popularity"].mean())

=== 1. Impacto del Contenido Explícito en el Mercado Activo ===
          popularity    energy   valence  dissonance  danceability
explicit                                                          
False      38.406487  0.638882  0.469106    0.169775      0.560117
True       42.135328  0.723716  0.465574    0.258142      0.626495

=== 2. Comparativa Estricta: Urbano vs Resto (Mercado Activo) ===
           popularity    energy   valence  dissonance  explicit  danceability
es_urbano                                                                    
False       38.694852  0.644427  0.465330    0.179098  0.082741      0.561452
True        40.051992  0.716881  0.607564    0.109317  0.225157      0.741213

=== 3. Relación de la Dissonancia con la Popularidad (Cuartiles) ===
dissonance_tier
Baja Tensión      37.613776
Media-Baja        39.398233
Media-Alta        40.865913
Máxima Tensión    37.035098
Name: popularity, dtype: float64
